# Stance sequential stability example

This notebook reruns the sequential Monte Carlo experiment for the full Stance population and computes the finite-population calibrated intervals during each replicate. The archived sequential CSV is kept under `Stance/archive/`, but it does not contain the sampling probabilities and selected-label masks needed for exact calibration. Historical estimator IDs (`spline`, `spline + tuning`) are displayed as `OPAL` and `OPAL + tuning` by the plotting helpers.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name in {"BRCA", "CheXpert", "Alphafold", "Stance"} else CWD
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import (
    EFFECTIVE_N_COL,
    HUMAN_N_COL,
    binary_odds_ratio_truth,
    run_sequential_odds_ratio_monte_carlo,
    summarize_monte_carlo,
)
from plotting import (
    METHOD_LABELS,
    make_monte_carlo_variance_table,
    plot_batch_sequential_coverage_comparison,
    plot_batch_sequential_effective_sample_size,
    plot_coverage,
    plot_effective_sample_size,
    plot_effective_sample_size_and_finite_population_coverage,
    plot_finite_population_coverage,
    plot_sequential_effective_sample_size,
    plot_sequential_effective_sample_size_distribution,
    plot_sequential_endpoint_variance_components,
    plot_sequential_variance_components,
    save_monte_carlo_variance_table,
)


In [2]:
EXAMPLE_DIR = REPO_ROOT / "Stance"
DATA_DIR = REPO_ROOT / "Data" / "Stance"
RESULTS_DIR = EXAMPLE_DIR / "results"
PLOTS_DIR = EXAMPLE_DIR / "plots"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 614
ALPHA = 0.10
TAU = 0.10
FRACS_HUMAN = np.linspace(0.2, 0.5, 10)
NUM_TRIALS = 500
BURNIN_STEPS = 50
RETRAIN_STEPS = 250
UNCERTAINTY_MODEL = "binned"
MAX_ITERATIONS = 50


## Load the fixed population

The sequential run evaluates against the entire cleaned Stance population, so the truth is the empirical odds ratio in this fixed population. This is the target for both ordinary coverage and finite-population calibrated coverage.

In [3]:
AFFIRMING_DEVICES = [
    "uncover", "realize", "know", "understand", "learn", "concede",
    "remember", "recall", "discover", "show", "reveal", "see",
    "forget", "find", "point out", "indicate", "acknowledge",
    "admit", "notice", "certify", "verify", "corroborate", "affirm",
    "confirm", "agree", "conclude", "proven", "settled", "conclusive",
    "definitive", "famed", "unequivocal", "skilful", "notable", "strong",
    "famous", "Nobel", "skillful", "Nobelist", "Nobel Laureate",
    "Nobel prize winner", "Nobel prize winning", "prize winning", "award",
    "winning", "distinguished", "well-grounded", "esteemed", "proficient",
    "key", "evidence", "noted", "top", "preeminent", "breakthrough",
    "significant", "intelligent", "of import", "celebrated", "novel", "recent",
    "major", "landmark", "important", "renowned", "peer-reviewed", "expert",
    "leading", "thousand", "1000", "hundred", "100", "unanimous", "diverse",
    "substantial", "many", "multiple", "dozen", "numerous",
]
LABEL_MAP = {"A": 1, "B": 0, "C": 0, "agrees": 1, "neutral": 0, "disagrees": 0}

raw_df = pd.read_csv(DATA_DIR / "stance_dataset.csv")
raw_df = raw_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
pattern = "|".join(re.escape(term) for term in AFFIRMING_DEVICES)
raw_df["contains_affirming_device"] = raw_df["sentence"].str.contains(
    pattern, case=False, regex=True
).fillna(False)

work = raw_df.dropna(subset=["confidence_in_prediction_gpt-4o", "label_gpt4o", "MACE_pred"]).copy()
work = work[
    work["label_gpt4o"].isin(LABEL_MAP)
    & work["MACE_pred"].isin(LABEL_MAP)
].copy()

Yhat = work["label_gpt4o"].map(LABEL_MAP).to_numpy(dtype=int)
Y = work["MACE_pred"].map(LABEL_MAP).to_numpy(dtype=int)
confidence = work["confidence_in_prediction_gpt-4o"].to_numpy(dtype=float).reshape(-1, 1)
device = work["contains_affirming_device"].to_numpy(dtype=bool)

true_odds_ratio, true_variance = binary_odds_ratio_truth(Y[~device], Y[device])
n_total = len(Y)

pd.DataFrame(
    {
        "group": ["no affirming device", "affirming device"],
        "n": [int((~device).sum()), int(device.sum())],
        "outcome_mean": [Y[~device].mean(), Y[device].mean()],
        "prediction_mean": [Yhat[~device].mean(), Yhat[device].mean()],
        "true_odds_ratio": [true_odds_ratio, true_odds_ratio],
        "true_variance": [true_variance, true_variance],
    }
)


,group,n,outcome_mean,prediction_mean,true_odds_ratio,true_variance
0,no affirming device,1833,0.38898,0.515003,0.798123,27.360524
1,affirming device,466,0.33691,0.510730,0.798123,27.360524


## Exact sequential Monte Carlo run

The run below records the usual confidence intervals and the finite-population calibrated intervals in the same pass. The exact calibration uses the realized selected-label masks and the first-order sampling probabilities from each sequential policy before those details are discarded.

In [4]:
seq_results = run_sequential_odds_ratio_monte_carlo(
    y=Y,
    yhat=Yhat,
    group1=device,
    uncertainty_features=confidence,
    fracs_human=FRACS_HUMAN,
    alpha=ALPHA,
    num_trials=NUM_TRIALS,
    true_odds_ratio=true_odds_ratio,
    true_variance=true_variance,
    burnin_steps=BURNIN_STEPS,
    retrain_steps=RETRAIN_STEPS,
    tau=TAU,
    seed=SEED,
    uncertainty_model=UNCERTAINTY_MODEL,
    show_progress=True,
)
seq_results = seq_results.sort_values([HUMAN_N_COL, "num_trial", "estimator"]).reset_index(drop=True)
seq_results.to_csv(RESULTS_DIR / "Stance_sequential_results.csv", index=False)

max_budget = float(seq_results[HUMAN_N_COL].max())
seq_results.head()


sequential human budget:   0%|          | 0/10 [00:00<?, ?it/s]

sequential trials 0.200:   0%|          | 0/500 [00:00<?, ?it/s]

/Users/ginniema/miniconda3/lib/python3.9/site-packages/cvxpy/problems/problem.py:1504: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


sequential trials 0.233:   0%|          | 0/500 [00:00<?, ?it/s]

sequential trials 0.267:   0%|          | 0/500 [00:00<?, ?it/s]

...

sequential trials 0.400:   0%|          | 0/500 [00:00<?, ?it/s]

sequential trials 0.433:   0%|          | 0/500 [00:00<?, ?it/s]

sequential trials 0.467:   0%|          | 0/500 [00:00<?, ?it/s]

sequential trials 0.500:   0%|          | 0/500 [00:00<?, ?it/s]

,trial,frac_human,$n_{\mathrm{human}}$,estimator,point estimate,log point estimate,variance estimate,lb,ub,interval width,...,mc finite_population_lb variance,mc finite_population_lb sd,mc finite_population_ub variance,mc finite_population_ub sd,mc finite_population_interval_width variance,mc finite_population_interval_width sd,mc variance,mc sd,mc log variance,mc log sd
0,0,0.2,459,active,1.076964,0.074146,105.314451,0.757371,1.531418,0.774047,...,0.019157,0.138409,0.031187,0.176598,0.003507,0.059216,0.025043,0.158251,0.041134,0.202816
1,0,0.2,459,active + tuning,1.028219,0.027828,86.966528,0.746704,1.415867,0.669164,...,0.014549,0.120621,0.027752,0.166589,0.003695,0.060786,0.020019,0.141488,0.033282,0.182432
2,0,0.2,459,classical,1.168242,0.155500,141.658986,0.776623,1.757339,0.980716,...,0.017699,0.133036,0.064595,0.254156,0.015259,0.123529,0.033695,0.183562,0.051592,0.227139
3,0,0.2,459,spline,0.599874,-0.511035,63.566075,0.456328,0.788576,0.332248,...,0.008733,0.093451,0.026080,0.161494,0.005413,0.073572,0.014901,0.122068,0.021903,0.147997
4,0,0.2,459,spline + tuning,0.608648,-0.496514,56.951729,0.469822,0.788497,0.318675,...,0.007237,0.085068,0.017828,0.133522,0.002662,0.051593,0.011281,0.106212,0.017205,0.131169


In [5]:
sequential_summary = summarize_monte_carlo(seq_results)
sequential_variance_table = make_monte_carlo_variance_table(seq_results)

sequential_summary.to_csv(RESULTS_DIR / "Stance_sequential_summary.csv", index=False)
sequential_variance_table.to_csv(RESULTS_DIR / "Stance_sequential_monte_carlo_variance_components.csv", index=False)
sequential_summary.head(12)


,$n_{\mathrm{human}}$,estimator,point_estimate_mean,point_estimate_var,log_point_estimate_mean,log_point_estimate_var,lb_mean,lb_var,ub_mean,ub_var,interval_width_var,coverage,finite_population_coverage,finite_population_interval_width,finite_population_variance_inflation,interval_width,variance_estimate,finite_population_log_variance_estimate,effective_n
0,459,active,0.805687,0.025043,-0.236022,0.041134,0.555835,0.016945,1.170324,0.035423,0.005039,0.954,0.906,0.534427,1.308742,0.614489,124.909426,0.042289,536.197416
1,459,active + tuning,0.786321,0.020019,-0.256752,0.033282,0.561955,0.012877,1.101372,0.031302,0.005385,0.950,0.902,0.453439,1.400872,0.539416,99.302405,0.031201,655.929880
2,459,classical,0.811966,0.033695,-0.233754,0.051592,0.542442,0.016374,1.215827,0.069815,0.019256,0.926,0.894,0.593941,1.286464,0.673385,140.257156,0.047946,452.454236
3,459,spline,0.800649,0.014901,-0.233433,0.021903,0.590688,0.007889,1.085492,0.028707,0.007093,0.954,0.902,0.396735,1.548151,0.494803,78.668347,0.022244,807.576483
4,459,spline + tuning,0.803417,0.011281,-0.227485,0.017205,0.608715,0.006430,1.060498,0.020013,0.004006,0.966,0.890,0.342322,1.729614,0.451783,65.495496,0.016533,965.259916
5,459,uniform,0.804826,0.026732,-0.237895,0.042163,0.556569,0.017921,1.165999,0.038334,0.005225,0.936,0.888,0.529001,1.312760,0.609430,122.753366,0.041353,541.600595
6,536,active,0.808807,0.021466,-0.228594,0.033134,0.574702,0.014909,1.139816,0.029909,0.003661,0.956,0.900,0.477744,1.385553,0.565114,104.162979,0.033302,632.588467
7,536,active + tuning,0.793022,0.015800,-0.244605,0.025771,0.580721,0.010427,1.083646,0.024198,0.003850,0.946,0.888,0.408895,1.498452,0.502926,84.607049,0.024844,763.207351
8,536,classical,0.801475,0.026862,-0.241935,0.041522,0.552684,0.013772,1.162539,0.052713,0.013071,0.944,0.898,0.532356,1.310172,0.609854,118.657302,0.039780,533.934121
9,536,spline,0.796348,0.010202,-0.235624,0.015789,0.602127,0.005591,1.053368,0.018950,0.004322,0.980,0.908,0.343757,1.715646,0.451240,66.393346,0.016942,953.994415


## Stability at the largest sequential budget

The line plots fix the largest human-label budget and show raw effective sample size across Monte Carlo replicate index. Because replicate order is arbitrary, the raw ESS distribution plot is the cleaner view for comparing variability across sequential runs.

In [6]:
plot_sequential_effective_sample_size(
    seq_results,
    path=PLOTS_DIR / "Stance_sequential_effective_sample_size_per_replicate_side_legend.pdf",
    n_human=max_budget,
    max_iterations=MAX_ITERATIONS,
    title="Effective Sample Size Across Sequential Runs",
    legend="side",
    show=False,
)

plot_sequential_effective_sample_size(
    seq_results,
    path=PLOTS_DIR / "Stance_sequential_effective_sample_size_per_replicate_no_legend.pdf",
    n_human=max_budget,
    max_iterations=MAX_ITERATIONS,
    title="Effective Sample Size Across Sequential Runs",
    legend="none",
    show=False,
)

plot_sequential_effective_sample_size_distribution(
    seq_results,
    path=PLOTS_DIR / "Stance_sequential_effective_sample_size_distribution.pdf",
    n_human=max_budget,
    title="Sequential Effective Sample Size Distribution",
    show=False,
)


(<Figure size 780x458 with 1 Axes>,
 <Axes: title={'center': 'Sequential Effective Sample Size Distribution'}, xlabel='Number of Effective Samples'>)

In [7]:
stability_table = (
    seq_results[seq_results[HUMAN_N_COL].eq(max_budget)]
    .groupby("estimator", observed=True)[EFFECTIVE_N_COL]
    .agg(ess_mean="mean", ess_sd="std", ess_min="min", ess_max="max", trials="count")
)
stability_table["ess_cv"] = stability_table["ess_sd"] / stability_table["ess_mean"]
stability_table = (
    stability_table.rename(index=lambda method: METHOD_LABELS.get(method, method))
    .sort_values("ess_mean", ascending=False)
)
stability_table.to_csv(RESULTS_DIR / "Stance_sequential_largest_budget_stability.csv")
stability_table


,ess_mean,ess_sd,ess_min,ess_max,trials,ess_cv
estimator,,,,,,
OPAL + tuning,1928.346402,28.650214,1833.010321,2009.237745,500,0.014857
OPAL,1849.084121,38.565598,1739.939523,1959.506531,500,0.020857
active + tuning,1473.482859,93.683172,1213.722533,1738.364363,500,0.063579
active,1336.574401,111.359356,1013.500159,1650.080510,500,0.083317
uniform,1258.392122,87.509950,1031.051119,1499.766206,500,0.069541
classical,1147.295372,54.763712,973.749394,1305.737569,500,0.047733


## Sequential budget plots

These plots show mean ESS with +/- 1 SD bars, a no-error-bar ESS version, ordinary empirical coverage, exact finite-population calibrated coverage, and across-run variance components. The sequential plot stays on the raw ESS scale; multiplier plots are only generated for the non-sequential example notebooks.

In [8]:
plot_effective_sample_size(
    seq_results,
    path=PLOTS_DIR / "Stance_sequential_effective_sample_size.pdf",
    n_total=n_total,
    error_bars="sd",
    show=False,
)

plot_effective_sample_size(
    seq_results,
    path=PLOTS_DIR / "Stance_sequential_effective_sample_size_no_error_bars.pdf",
    n_total=n_total,
    error_bars="none",
    show=False,
)

plot_effective_sample_size_and_finite_population_coverage(
    seq_results,
    alpha=ALPHA,
    path=PLOTS_DIR / "Stance_sequential_effective_sample_size_and_corrected_coverage.pdf",
    n_total=n_total,
    show=False,
)

plot_effective_sample_size_and_finite_population_coverage(
    seq_results,
    alpha=ALPHA,
    path=PLOTS_DIR / "Stance_sequential_effective_sample_size_and_corrected_coverage.png",
    n_total=n_total,
    show=False,
)

plot_coverage(
    seq_results,
    alpha=ALPHA,
    path=PLOTS_DIR / "Stance_sequential_coverage.pdf",
    n_total=n_total,
    show=False,
)

plot_finite_population_coverage(
    seq_results,
    alpha=ALPHA,
    path=PLOTS_DIR / "Stance_sequential_coverage_finite_population_calibrated.pdf",
    n_total=n_total,
    show=False,
)

batch_results = pd.read_csv(RESULTS_DIR / "Stance_results.csv")
batch_n_total = int(batch_results[HUMAN_N_COL].max() / batch_results["frac_human"].max())

plot_batch_sequential_effective_sample_size(
    batch_results,
    seq_results,
    path=PLOTS_DIR / "Stance_batch_sequential_effective_sample_size.pdf",
    n_total_batch=batch_n_total,
    n_total_sequential=n_total,
    show=False,
)
plot_batch_sequential_effective_sample_size(
    batch_results,
    seq_results,
    path=PLOTS_DIR / "Stance_batch_sequential_effective_sample_size.png",
    n_total_batch=batch_n_total,
    n_total_sequential=n_total,
    show=False,
)
plot_batch_sequential_coverage_comparison(
    batch_results,
    seq_results,
    alpha=ALPHA,
    path=PLOTS_DIR / "Stance_batch_sequential_coverage_comparison.pdf",
    n_total_batch=batch_n_total,
    n_total_sequential=n_total,
    show=False,
)
plot_batch_sequential_coverage_comparison(
    batch_results,
    seq_results,
    alpha=ALPHA,
    path=PLOTS_DIR / "Stance_batch_sequential_coverage_comparison.png",
    n_total_batch=batch_n_total,
    n_total_sequential=n_total,
    show=False,
)

plot_sequential_variance_components(
    seq_results,
    path=PLOTS_DIR / "Stance_sequential_variance_components.pdf",
    n_total=n_total,
    show=False,
)

plot_sequential_endpoint_variance_components(
    seq_results,
    path=PLOTS_DIR / "Stance_sequential_endpoint_variance_components.pdf",
    n_total=n_total,
    show=False,
)
plot_sequential_endpoint_variance_components(
    seq_results,
    path=PLOTS_DIR / "Stance_sequential_endpoint_variance_components.png",
    n_total=n_total,
    show=False,
)

save_monte_carlo_variance_table(
    seq_results,
    path=PLOTS_DIR / "Stance_sequential_monte_carlo_variance_table.pdf",
    max_rows=18,
    show=False,
)


(<Figure size 1400x532 with 1 Axes>, <Axes: >)